# The Most Beautiful Trick in DeepSeek's V4 Paper
## Part 1: The SVD and Muon

Two ideas, one trick:

1. **Newton-Schulz iteration** — compute $VW^T$ (the polar factor) without ever running the SVD. Just matrix multiplies.
2. **Muon optimizer** — use that trick to orthogonalize momentum before each weight update.

All experiments log to **W&B** (`mhc-nanogpt-blog`).

In [ ]:
!pip install -q git+https://github.com/ashishjv1/mHC.git
!pip install -q wandb

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import wandb

plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

from src.newton_schulz import newton_schulz_muon
from src.muon import Muon
from src.model import GPT
from configs.train_config import TrainConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass
wandb.login()

---
## Part 1: The $VW^T$ Trick

Any matrix $G$ has an SVD: $G = V \Sigma W^T$. The **polar factor** $U = VW^T$ keeps the rotation but throws away the scaling — it sets every singular value to 1.

Computing the SVD costs $O(n^3)$ and is sequential. Newton-Schulz computes $VW^T$ iteratively using only matmuls:

$$X_{k+1} = X_k \left( aI + b X_k^T X_k + c (X_k^T X_k)^2 \right)$$

Each iteration maps singular values through the polynomial $p(\sigma) = \sigma(a + b\sigma^2 + c\sigma^4)$. Pick coefficients where $\sigma = 1$ is the stable fixed point, and after ~5 iterations every singular value has been driven to 1. You get $VW^T$ without ever computing $V$, $\Sigma$, or $W$ individually.

**Why this matters for training:** the gradient matrix $\nabla W$ has a wildly spread spectrum — some directions carry 1000x more signal than others. Replacing $\nabla W$ with its polar factor $VW^T$ equalizes all directions. The optimizer sees a landscape where every direction matters equally.

### Watching it happen: SVs of a random matrix under Newton-Schulz

In [ ]:
run = wandb.init(project='mhc-nanogpt-blog', name='ns_svd_convergence',
                 tags=['newton-schulz', 'blog-part1'])

torch.manual_seed(42)
G = torch.randn(64, 64)
max_iters = 12

# Polar coefficients: a+b+c = 1, so σ=1 is the fixed point
a, b, c = 15/8, -10/8, 3/8

X = G.float() / torch.linalg.norm(G.float(), ord=2)  # spectral normalize

sv_history = [torch.linalg.svdvals(X).cpu().numpy()]
for _ in range(max_iters):
    A = X @ X.T
    B = A @ X
    X = a * X + b * B + c * A @ B
    sv_history.append(torch.linalg.svdvals(X).cpu().numpy())

sv_arr = np.array(sv_history)

fig, ax = plt.subplots(figsize=(10, 5))
for j in range(sv_arr.shape[1]):
    ax.plot(range(max_iters + 1), sv_arr[:, j], alpha=0.4, linewidth=0.8)
ax.axhline(y=1.0, color='red', ls='--', linewidth=2, label='target: all $\\sigma$ = 1 ($VW^T$)')
ax.set_xlabel('Newton-Schulz Iteration')
ax.set_ylabel('Singular Value')
ax.set_title('64 singular values of a random matrix converging to 1.0\n$G \\to VW^T$ without computing the SVD')
ax.legend(fontsize=11)
plt.tight_layout()
wandb.log({'ns_convergence': wandb.Image(fig)})
plt.show()

for i, svs in enumerate(sv_history):
    wandb.log({'ns_iter': i, 'sv/min': svs.min(), 'sv/max': svs.max(),
               'sv/std': svs.std(), 'sv/condition': svs.max() / (svs.min() + 1e-8)})

print(f'After {max_iters} iterations: SV range [{sv_history[-1].min():.6f}, {sv_history[-1].max():.6f}]')
wandb.finish()

---
## Part 2: Muon — Using $VW^T$ to Train Better

Muon = **M**omentum orthogonalized by Newton-Schulz.

Each step:
1. Update momentum buffer: $m \leftarrow \beta m + g$
2. Orthogonalize: $\hat{m} = \text{NS}(m) \approx VW^T$ of $m$
3. Update weights: $w \leftarrow w - \eta \cdot \hat{m}$

The gradient's spectrum can have condition numbers of 100-10,000x. After NS, the update's singular values are all equalized — the optimizer steps equally in every direction. This makes the loss landscape look isotropic to the optimizer, even when it isn't.

Let's train a small transformer on **WikiText-2** and compare AdamW vs Muon.

In [ ]:
import tiktoken
from datasets import load_dataset
import math

# Prepare WikiText-2
print('Downloading WikiText-2...')
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train', trust_remote_code=True)
enc = tiktoken.get_encoding('gpt2')

all_tokens = []
for row in ds:
    text = row['text'].strip()
    if text:
        all_tokens.extend(enc.encode_ordinary(text))
        all_tokens.append(enc.eot_token)
all_tokens = torch.tensor(all_tokens, dtype=torch.long)
print(f'WikiText-2: {len(all_tokens):,} tokens')


def get_batch(tokens, batch_size, ctx_len):
    ix = torch.randint(len(tokens) - ctx_len - 1, (batch_size,))
    x = torch.stack([tokens[i:i+ctx_len] for i in ix]).to(device)
    y = torch.stack([tokens[i+1:i+1+ctx_len] for i in ix]).to(device)
    return x, y

In [ ]:
def train_run(optimizer_name, tokens, run_name):
    """Train a GPT on WikiText-2, return loss history."""
    cfg = TrainConfig(
        n_layers=6, d_model=512, n_heads=8, d_ff=2048,
        vocab_size=50304, context_len=256,
        dropout=0.0, bias=False, use_mhc=False, compile=False,
        optimizer=optimizer_name,
        lr=6e-4, min_lr=6e-5,
        muon_lr=0.02, muon_min_lr=0.002, muon_momentum=0.95, muon_ns_steps=5,
        weight_decay=0.1, batch_size=32, grad_accum_steps=1,
        max_steps=1000, warmup_steps=100,
    )

    torch.manual_seed(42)
    wandb.init(project='mhc-nanogpt-blog', name=run_name,
               tags=['training', 'blog-part1', optimizer_name],
               config={'optimizer': optimizer_name, 'dataset': 'wikitext-2',
                       'd_model': cfg.d_model, 'n_layers': cfg.n_layers,
                       'max_steps': cfg.max_steps})

    model = GPT(cfg).to(device)
    print(f'{run_name}: {model.count_parameters():,} params')

    if optimizer_name == 'adamw':
        decay = [p for _, p in model.named_parameters() if p.requires_grad and p.ndim >= 2]
        nodecay = [p for _, p in model.named_parameters() if p.requires_grad and p.ndim < 2]
        opt = torch.optim.AdamW([
            {'params': decay, 'weight_decay': cfg.weight_decay},
            {'params': nodecay, 'weight_decay': 0.0},
        ], lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
        muon_opt = None
    else:
        muon_params, adamw_decay, adamw_nodecay = model.get_param_groups()
        muon_opt = Muon([{'params': muon_params}], lr=cfg.muon_lr,
                        momentum=cfg.muon_momentum, ns_steps=cfg.muon_ns_steps)
        opt = torch.optim.AdamW([
            {'params': adamw_decay, 'weight_decay': cfg.weight_decay},
            {'params': adamw_nodecay, 'weight_decay': 0.0},
        ], lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))

    losses = []
    for step in range(cfg.max_steps):
        if step < cfg.warmup_steps:
            lr = cfg.lr * (step + 1) / cfg.warmup_steps
        else:
            progress = (step - cfg.warmup_steps) / (cfg.max_steps - cfg.warmup_steps)
            lr = cfg.min_lr + 0.5 * (cfg.lr - cfg.min_lr) * (1 + math.cos(math.pi * progress))
        for g in opt.param_groups:
            g['lr'] = lr
        if muon_opt is not None:
            if step < cfg.warmup_steps:
                mlr = cfg.muon_lr * (step + 1) / cfg.warmup_steps
            else:
                mlr = cfg.muon_min_lr + 0.5 * (cfg.muon_lr - cfg.muon_min_lr) * (
                    1 + math.cos(math.pi * (step - cfg.warmup_steps) / (cfg.max_steps - cfg.warmup_steps)))
            for g in muon_opt.param_groups:
                g['lr'] = mlr

        x, y = get_batch(tokens, cfg.batch_size, cfg.context_len)
        model.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

        if muon_opt is not None:
            muon_opt.step()
        opt.step()

        losses.append(loss.item())
        wandb.log({'train/loss': loss.item(), 'train/lr': lr}, step=step)
        if step % 200 == 0:
            print(f'  step {step:>4d} | loss {loss.item():.4f}')

    wandb.finish()
    return losses

In [ ]:
print('=== AdamW ===')
adamw_losses = train_run('adamw', all_tokens, 'demo_adamw')

print('\n=== Muon ===')
muon_losses = train_run('muon', all_tokens, 'demo_muon')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

w = 20
ax.plot(np.convolve(adamw_losses, np.ones(w)/w, mode='valid'),
        color='#1f77b4', linewidth=1.5, label='AdamW', alpha=0.9)
ax.plot(np.convolve(muon_losses, np.ones(w)/w, mode='valid'),
        color='#ff7f0e', linewidth=1.5, label='Muon', alpha=0.9)
ax.set_xlabel('Step')
ax.set_ylabel('Loss (smoothed)')
ax.set_title('AdamW vs Muon on WikiText-2 (6-layer, 512-dim, ~45M params)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'AdamW final loss (last 50 steps): {np.mean(adamw_losses[-50:]):.4f}')
print(f'Muon  final loss (last 50 steps): {np.mean(muon_losses[-50:]):.4f}')

---
## What's Next

Part 2 of the blog covers the **manifold constraint** — using the same Newton-Schulz trick to keep routing matrices on the orthogonal group during training. Same polynomial, different purpose.

All plots are on W&B under project `mhc-nanogpt-blog`.